In [ ]:
# 1. INSTALLATION
!pip install catboost==1.2.8 scikit-learn==1.6.1 ta==0.11.0 pandas_datareader==0.10.0 tabulate requests > /dev/null 2>&1

# 2. DRIVE MOUNT & PATH CONFIG
import os
import sys
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Global_Macro_Model'
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)

os.makedirs('artifacts', exist_ok=True)
os.makedirs('data_cache', exist_ok=True)

# 3. CORE LOGIC
import time
import requests
import warnings
import joblib
import logging
import pandas as pd
import numpy as np
from datetime import datetime
from pandas_datareader import data as pdr
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
try:
    from zoneinfo import ZoneInfo
except ImportError:
    from backports.zoneinfo import ZoneInfo

# --- SILENCE WARNINGS ---
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message=".*cv='prefit'.*")
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
logging.getLogger('numexpr').setLevel(logging.CRITICAL)

# --- CONFIGURATION (SPY ONLY) ---
INITIAL_CAPITAL = 2500
FAST_MODE = True 

ASSET_CONFIG = {
    'SPY': {
        'target_pct': 0.0025,
        'win_amount': 55,
        'loss_amount': 45,
        'description': 'S&P 500',
    }
}

TWELVEDATA_API_KEY = os.getenv("TWELVEDATA_API_KEY", "7c5a55559d164d4ebca0d23a5baea839").strip()

# --- DATA ENGINE ---
def download_fred_series(series, start):
    try:
        df = pdr.DataReader(series, "fred", start)
        return df.dropna()
    except: return pd.DataFrame()

def _fred_to_ohlcv(series_df: pd.DataFrame, col: str) -> pd.DataFrame:
    if series_df.empty: return pd.DataFrame()
    s = series_df[col]
    df = pd.DataFrame(index=s.index)
    df['Close'] = s; df['Open'] = s; df['High'] = s; df['Low'] = s; df['Volume'] = 0
    if df.index.tz is not None: df.index = df.index.tz_localize(None)
    return df

def download_twelvedata_robust(symbol, api_key, limit=5000):
    if not api_key: return pd.DataFrame()
    url = "https://api.twelvedata.com/time_series"
    params = {'symbol': symbol, 'interval': '1day', 'outputsize': limit, 'format': 'JSON', 'apikey': api_key, 'order': 'ASC'}
    try:
        for attempt in range(3):
            r = requests.get(url, params=params, timeout=45)
            if r.status_code == 200: break
            time.sleep(2)
        if r.status_code != 200: return pd.DataFrame()
        j = r.json()
        if 'values' not in j: return pd.DataFrame()
        df = pd.DataFrame(j['values'])
        df['datetime'] = pd.to_datetime(df['datetime'])
        df = df.set_index('datetime').sort_index()
        rename = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}
        df = df.rename(columns=rename)
        for col in ['Open', 'High', 'Low', 'Close']: df[col] = pd.to_numeric(df[col], errors='coerce')
        df['Volume'] = pd.to_numeric(df.get('Volume', 0), errors='coerce').fillna(0)
        if df.index.tz is not None: df.index = df.index.tz_localize(None)
        return df[['Open', 'High', 'Low', 'Close', 'Volume']]
    except: return pd.DataFrame()

def prepare_data(ticker, config, is_inference=False):
    limit = 750 if is_inference else 5000
    print(f"Downloading data for {ticker} ({'Inference' if is_inference else 'Training'})...", end=" ", flush=True)
    asset = download_twelvedata_robust(ticker, TWELVEDATA_API_KEY, limit=limit)
    
    if len(asset) < (200 if is_inference else 3000):
        print("FAILED.")
        raise ValueError(f"CRITICAL: Insufficient data for {ticker}. Got {len(asset)} rows.")

    start_dt = asset.index.min()
    vix = download_fred_series("VIXCLS", start=start_dt)
    vix = _fred_to_ohlcv(vix, 'VIXCLS')
    tnx = download_fred_series("DGS10", start=start_dt)
    tnx = _fred_to_ohlcv(tnx, 'DGS10')
    
    if not is_inference: time.sleep(8) 
    hyg = download_twelvedata_robust("HYG", TWELVEDATA_API_KEY, limit=limit)
    if not is_inference: time.sleep(8) 
    xlk = download_twelvedata_robust("XLK", TWELVEDATA_API_KEY, limit=limit)
    
    closes = pd.DataFrame(index=asset.index)
    closes[ticker] = asset["Close"]
    closes["^VIX"] = vix["Close"].reindex(closes.index).ffill().bfill()
    closes["^TNX"] = tnx["Close"].reindex(closes.index).ffill().bfill()
    closes["HYG"] = hyg["Close"].reindex(closes.index).ffill().bfill()
    closes["XLK"] = xlk["Close"].reindex(closes.index).ffill().bfill()
    
    asset_open = asset["Open"]; asset_close = asset["Close"]
    asset_high = asset["High"]; asset_low = asset["Low"]
    asset_volume = asset["Volume"]
    vix_open = vix["Open"].reindex(closes.index).ffill().bfill()
    vix_close = vix["Close"].reindex(closes.index).ffill().bfill()
    
    _df = closes.copy()
    base_tickers = [ticker, '^VIX', '^TNX', 'HYG', 'XLK']
    for t in base_tickers:
        if t in _df.columns: _df[f'{t}_Log_Return_Close'] = np.log(_df[t] / _df[t].shift(1)).shift(1)
    
    _df[f'{ticker}_Overnight_Gap'] = (asset_open - asset_close.shift(1)) / asset_close.shift(1)
    _df['VIX_Overnight_Gap'] = (vix_open - vix_close.shift(1)) / vix_close.shift(1)
    
    lags = [1, 2, 3, 5, 10]
    for lag in lags:
        _df[f'{ticker}_Return_Lag{lag}'] = _df[f'{ticker}_Log_Return_Close'].shift(lag)
        _df[f'VIX_Return_Lag{lag}'] = _df['^VIX_Log_Return_Close'].shift(lag)
    
    for w in [5, 10, 20, 60]: _df[f'{ticker}_Roll_Std_{w}'] = _df[f'{ticker}_Log_Return_Close'].rolling(window=w).std()
        
    close_s = _df[ticker]
    sma20 = close_s.rolling(20).mean().shift(1)
    sma200 = close_s.rolling(200).mean().shift(1)
    sma50 = close_s.rolling(50).mean().shift(1)
    
    _df[f'{ticker}_SMA20_Dist'] = (close_s.shift(1) / sma20) - 1
    _df[f'{ticker}_SMA50_Dist'] = (close_s.shift(1) / sma50) - 1
    _df[f'{ticker}_SMA200_Dist'] = (close_s.shift(1) / sma200) - 1
    _df[f'{ticker}_SMA20_50_Spread'] = (sma20 / sma50) - 1
    _df[f'{ticker}_RSI'] = RSIIndicator(close=close_s, window=14).rsi().shift(1)
    macd_obj = MACD(close=close_s, window_slow=26, window_fast=12, window_sign=9)
    _df['MACD_12_26_9'] = macd_obj.macd().shift(1)
    _df['MACDs_12_26_9'] = macd_obj.macd_signal().shift(1)
    bb = BollingerBands(close=close_s, window=20, window_dev=2)
    bb_width = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()
    _df[f'{ticker}_BB_Width'] = bb_width.shift(1)
    atr = AverageTrueRange(high=asset_high, low=asset_low, close=asset_close, window=14).average_true_range()
    _df[f'{ticker}_ATR'] = atr.shift(1)
    price_change = asset_close.diff()
    direction = np.sign(price_change).fillna(0)
    obv = (direction * asset_volume).cumsum()
    _df[f'{ticker}_OBV'] = obv.shift(1)
    
    _df['Day_of_Week'] = _df.index.dayofweek
    _df['Day_of_Month'] = _df.index.day
    
    target_pct = config['target_pct']
    intraday_change = (asset_close - asset_open) / asset_open
    conditions = [(intraday_change > target_pct), (intraday_change < -target_pct)]
    choices = [1, -1]
    _df['Target'] = np.select(conditions, choices, default=0)
    
    _df.dropna(inplace=True)
    _df.index = _df.index.tz_localize(None)
    print("Done.")
    return _df

# --- MODEL UTILS ---
def apply_rule_from_probs(probs, threshold, max_noise, margin):
    bear_p = probs[:, 0]; noise_p = probs[:, 1]; bull_p = probs[:, 2]
    pred = np.ones(len(probs), dtype=int)
    bear_mask = (bear_p > threshold)
    bull_mask = (bull_p > threshold)
    if max_noise < 1.0:
        bear_mask = bear_mask & (noise_p <= max_noise)
        bull_mask = bull_mask & (noise_p <= max_noise)
    pred[bear_mask] = 0; pred[bull_mask] = 2
    return pred

def make_base_model(fast):
    return CatBoostClassifier(
        loss_function='MultiClass', iterations=600 if fast else 1500, 
        depth=6, learning_rate=0.05, l2_leaf_reg=6, 
        random_seed=42, allow_writing_files=False, logging_level='Silent'
    )

def fit_model_full(X, y_mapped, fast):
    n = len(X); n_cal = max(60, int(n * 0.15))
    X_fit, y_fit = X[:-n_cal], y_mapped[:-n_cal]
    X_cal, y_cal = X[-n_cal:], y_mapped[-n_cal:]
    base = make_base_model(fast)
    base.fit(X_fit, y_fit)
    cal = CalibratedClassifierCV(base, method='sigmoid', cv='prefit')
    cal.fit(X_cal, y_cal)
    return cal

def oof_probs_tabular(X, y_mapped, fast):
    n_splits = 3 if fast else 5
    tscv = TimeSeriesSplit(n_splits=n_splits)
    oof = np.full((len(X), 3), np.nan, dtype=float)
    for tr_idx, val_idx in tscv.split(X):
        model = fit_model_full(X[tr_idx], y_mapped[tr_idx], fast=fast)
        oof[val_idx] = model.predict_proba(X[val_idx])
    mask = ~np.isnan(oof).any(axis=1)
    return oof[mask], mask

def optimize_rule_on_train(probs_train, true_target_train, win_amount, loss_amount, min_trades=20):
    threshold_grid = np.arange(0.35, 0.85, 0.02)
    max_noise_grid = [1.00, 0.80, 0.60]
    best = {'threshold': 0.60, 'max_noise': 1.00, 'margin': 0.00, 'ev_total': -1e9}
    
    for max_noise in max_noise_grid:
        for th in threshold_grid:
            pred = apply_rule_from_probs(probs_train, th, max_noise, margin=0.0)
            trade_mask = pred != 1
            if np.sum(trade_mask) < min_trades: continue
            
            p_dir = np.where(pred[trade_mask] == 0, -1, 1)
            actual = true_target_train[trade_mask]
            wins = np.sum(p_dir == actual)
            losses = len(p_dir) - wins
            ev = wins * win_amount - losses * loss_amount
            if ev > best['ev_total']:
                 best = {'threshold': float(th), 'max_noise': float(max_noise), 'margin': 0.0, 'ev_total': ev}
    return best

# ------------------------------------------------------------
# WEEKEND TRAINING (VALIDATION + SAVE)
# ------------------------------------------------------------
def run_weekend_training_mode():
    print(f"\n{'='*70}")
    print(f"  WEEKEND TRAINING & HEALTH CHECK (SPY ONLY)")
    print(f"{'='*70}")
    
    for ticker, config in ASSET_CONFIG.items():
        try:
            # 1. DATA
            df = prepare_data(ticker, config, is_inference=False)
            
            exclude = ['Target', ticker, '^VIX', '^TNX', 'HYG', 'XLK']
            feats = [c for c in df.columns if c not in exclude]
            X_all = df[feats].values
            y_all = (df['Target'].values + 1).astype(int)
            y_targets = df['Target'].values
            dates = df.index
            
            print(f"\n--- {ticker} PHASE 1: LONG-TERM STABILITY (2015-2024) ---")
            
            sim_start_date = pd.Timestamp("2015-01-01")
            start_idx = df.index.searchsorted(sim_start_date)
            if start_idx < 500: start_idx = 500
            
            current_idx = start_idx
            cutoff_1year = df.index.searchsorted(pd.Timestamp.now() - pd.Timedelta(days=365))
            step_months = 3
            
            # PHASE 1 LOOP
            while current_idx < cutoff_1year:
                end_idx = min(current_idx + 63, cutoff_1year) # ~3 months
                if end_idx == current_idx: break
                
                X_tr = X_all[:current_idx]; y_tr = y_all[:current_idx]
                X_te = X_all[current_idx:end_idx]; y_te_targ = y_targets[current_idx:end_idx]
                
                oof, mask = oof_probs_tabular(X_tr, y_tr, fast=FAST_MODE)
                rule = optimize_rule_on_train(oof, y_targets[:current_idx][mask], config['win_amount'], config['loss_amount'])
                model = fit_model_full(X_tr, y_tr, fast=FAST_MODE)
                
                probs = model.predict_proba(X_te)
                pred = apply_rule_from_probs(probs, rule['threshold'], rule['max_noise'], rule['margin'])
                
                n_trades = np.sum(pred != 1)
                n_wins = 0
                if n_trades > 0:
                    p_dirs = np.where(pred[pred!=1]==0, -1, 1)
                    acts = y_te_targ[pred!=1]
                    n_wins = np.sum(p_dirs == acts)
                    wr = (n_wins / n_trades) * 100
                else: wr = 0.0
                
                noise_pct = (1.0 - (n_trades/len(pred))) * 100
                
                d_str = f"{dates[current_idx].date()} -> {dates[end_idx-1].date()}"
                print(f"  {d_str} | Noise: {noise_pct:3.0f}% | Trades: {n_trades:2d} | WinRate: {wr:5.1f}%")
                
                current_idx = end_idx

            print(f"\n--- {ticker} PHASE 2: LAST 52 WEEKS (Micro Review) ---")
            
            step_week = 5
            total_len = len(X_all)
            current_idx = cutoff_1year 
            
            while current_idx < total_len:
                end_idx = min(current_idx + step_week, total_len)
                if end_idx == current_idx: break
                
                X_tr = X_all[:current_idx]; y_tr = y_all[:current_idx]
                X_te = X_all[current_idx:end_idx]; y_te_targ = y_targets[current_idx:end_idx]
                
                oof, mask = oof_probs_tabular(X_tr, y_tr, fast=FAST_MODE)
                rule = optimize_rule_on_train(oof, y_targets[:current_idx][mask], config['win_amount'], config['loss_amount'])
                model = fit_model_full(X_tr, y_tr, fast=FAST_MODE)
                
                probs = model.predict_proba(X_te)
                pred = apply_rule_from_probs(probs, rule['threshold'], rule['max_noise'], rule['margin'])
                
                n_trades = np.sum(pred != 1)
                n_wins = 0
                if n_trades > 0:
                    p_dirs = np.where(pred[pred!=1]==0, -1, 1)
                    acts = y_te_targ[pred!=1]
                    n_wins = np.sum(p_dirs == acts)
                    wr = (n_wins / n_trades) * 100
                else: wr = 0.0
                
                noise_pct = (1.0 - (n_trades/len(pred))) * 100
                d_str = f"{dates[current_idx].date()} -> {dates[end_idx-1].date()}"
                print(f"  {d_str} | Noise: {noise_pct:3.0f}% | Trades: {n_trades} | WinRate: {wr:5.1f}%")
                
                current_idx = end_idx

            # PHASE 3: FINAL TRAIN
            print(f"\n[{ticker}] Training Final Model on Full History...")
            final_model = fit_model_full(X_all, y_all, fast=FAST_MODE)
            oof_probs, mask = oof_probs_tabular(X_all, y_all, fast=FAST_MODE)
            final_rule = optimize_rule_on_train(oof_probs, y_targets[mask], config['win_amount'], config['loss_amount'], min_trades=30)
            
            print(f"[{ticker}] SAVING ARTIFACTS: Th={final_rule['threshold']:.2f}, Noise={final_rule['max_noise']:.2f}")
            
            save_dir = os.path.join('artifacts', ticker)
            os.makedirs(save_dir, exist_ok=True)
            joblib.dump(final_model, os.path.join(save_dir, 'tabular_model.pkl'))
            joblib.dump(feats, os.path.join(save_dir, 'feature_cols.pkl'))
            with open(os.path.join(save_dir, 'threshold.txt'), 'w') as f: f.write(str(final_rule['threshold']))
            with open(os.path.join(save_dir, 'max_noise_prob.txt'), 'w') as f: f.write(str(final_rule['max_noise']))
            with open(os.path.join(save_dir, 'margin_vs_noise.txt'), 'w') as f: f.write(str(final_rule['margin']))
            
        except Exception as e:
            print(f"ERROR training {ticker}: {e}")

# ------------------------------------------------------------
# 5. INFERENCE FUNCTION (Run Daily)
# ------------------------------------------------------------
def run_inference_mode():
    now_et = datetime.now(ZoneInfo("America/New_York"))
    print(f"\n{'#'*60}")
    print(f"  LIVE INFERENCE - {now_et.strftime('%Y-%m-%d %H:%M')} ET")
    print(f"{'#'*60}")
    
    for ticker in ASSET_CONFIG.keys():
        save_dir = os.path.join('artifacts', ticker)
        if not os.path.exists(save_dir): continue
            
        try:
            model = joblib.load(os.path.join(save_dir, 'tabular_model.pkl'))
            feats = joblib.load(os.path.join(save_dir, 'feature_cols.pkl'))
            with open(os.path.join(save_dir, 'threshold.txt'), 'r') as f: th = float(f.read().strip())
            with open(os.path.join(save_dir, 'max_noise_prob.txt'), 'r') as f: mn = float(f.read().strip())
            with open(os.path.join(save_dir, 'margin_vs_noise.txt'), 'r') as f: mg = float(f.read().strip())
            
            df = prepare_data(ticker, ASSET_CONFIG[ticker], is_inference=True)
            if len(df) == 0: continue
            
            X = np.zeros((len(df), len(feats)))
            for i, col in enumerate(feats):
                if col in df.columns: X[:, i] = df[col].values
            
            probs = model.predict_proba(X)
            pred = apply_rule_from_probs(probs, th, mn, mg)
            
            latest_idx = -1
            latest_date = df.index[latest_idx]
            latest_probs = probs[latest_idx]
            latest_sig = pred[latest_idx]
            
            sig_str = {0: 'BEAR', 1: 'NOISE', 2: 'BULL'}[latest_sig]
            
            print(f"\n>>> {ticker} SIGNAL ({latest_date.date()}) <<<")
            print(f"Prediction: {sig_str}")
            print(f"Probabilities: Bear={latest_probs[0]:.1%} | Noise={latest_probs[1]:.1%} | Bull={latest_probs[2]:.1%}")
            
            if latest_sig == 2: print(f"ACTION: BUY CALL DEBIT SPREAD")
            elif latest_sig == 0: print(f"ACTION: BUY PUT DEBIT SPREAD")
            else: print("ACTION: WAIT / NO TRADE")
                
        except Exception as e:
            print(f"Error inference {ticker}: {e}")

# ============================================================
# EXECUTION CONTROL
# ============================================================

# 1. WEEKEND: Uncomment to Train (Takes time, verifies history)
#run_weekend_training_mode()

# 2. WEEKDAY: Uncomment to Trade (Fast)
run_inference_mode()